# Laboratorio 2A — Carga de datos y manejo de DataFrames

Antes de auditar un conjunto de datos hay que poder abrirlo, describirlo y recortarlo con
soltura. Este laboratorio fija ese piso: **cargar un archivo, entender qué trajo, seleccionar,
filtrar, agrupar y resumir**. No hay algoritmos acá; hay manejo de la herramienta con la que se
trabaja el resto del curso.

El conjunto de datos es real: **Heart Failure Prediction**, 918 pacientes y 12 variables
clínicas, publicado sin credencialización. Se carga directamente desde su URL, así que el
notebook abre en Google Colab sin subir nada.

Las celdas que contienen `raise NotImplementedError` deben completarse.

## 0. Preparación

`pandas` y `matplotlib` vienen instalados en Colab. La única dependencia externa es la conexión
para descargar el archivo.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 1. Cargar un CSV

`pd.read_csv` acepta una ruta local o una URL. En un archivo local sería
`pd.read_csv("datos/heart.csv")`; acá se lee directo del repositorio público.

In [4]:
URL = "https://raw.githubusercontent.com/gustavovazquez/datasets/main/heart.csv"

df = pd.read_csv(URL)
df.shape

(918, 12)

Los argumentos de `read_csv` que más se usan en la práctica:

| Argumento | Para qué |
|---|---|
| `sep` | Separador, cuando no es la coma (`sep=";"` es habitual en archivos europeos) |
| `decimal` | Separador decimal (`decimal=","`) |
| `na_values` | Qué cadenas se leen como faltante (`na_values=["", "NA", "sin dato"]`) |
| `dtype` | Forzar el tipo de una columna (`dtype={"id": str}`, para no perder ceros a la izquierda) |
| `usecols` | Leer solo algunas columnas, cuando el archivo es grande |
| `nrows` | Leer las primeras filas, para inspeccionar antes de cargar todo |

**Un identificador se lee siempre como texto.** Si se lee como entero, `00734` se convierte en
`734` y el identificador queda destruido.

## 2. Primer vistazo

Las cuatro operaciones que se hacen siempre, en este orden: forma, primeras filas, tipos y
resumen numérico.

In [5]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    str    
 2   ChestPainType   918 non-null    str    
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    str    
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    str    
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    str    
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 86.2 KB


In [7]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


`describe()` solo resume las columnas numéricas. Para las categóricas hay que pedirlo
explícitamente.

In [8]:
df.describe(include="object")

/tmp/ipykernel_51048/702825166.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,Sex,ChestPainType,RestingECG,ExerciseAngina,ST_Slope
count,918,918,918,918,918
unique,2,4,3,2,3
top,M,ASY,Normal,N,Flat
freq,725,496,552,547,460


### Para analizar

`RestingBP` (presión en reposo) y `Cholesterol` tienen mínimo 0. Un colesterol de 0 mg/dl no es
una medición **posible**: es un código que la institución usó para «no medido».

Responder a partir de la salida de `describe()`:

1. ¿Cuántos registros tienen `Cholesterol == 0`? ¿Qué porcentaje del total representan?
2. Si esos ceros se dejan como están, ¿en qué dirección se desplaza la media de `Cholesterol`?

In [9]:
# PREGUNTA 1: 
cantidad=(df['Cholesterol']==0).sum()
print(cantidad)
pcte=(cantidad/len(df))*100
print(pcte)




172
18.736383442265794


**Respuesta:**

2. SE DESPLAZARIA HACIA ABAJO SI ESTUVIERAMOS HABLANDO DE VALORES REALES O UN GRAFICO.

## 3. Selección de columnas y filas

Tres formas que conviene no mezclar:

| Sintaxis | Qué devuelve |
|---|---|
| `df["Age"]` | Una **Series** (una columna) |
| `df[["Age", "Sex"]]` | Un **DataFrame** (lista de columnas) |
| `df.loc[filas, columnas]` | Selección **por etiqueta** |
| `df.iloc[filas, columnas]` | Selección **por posición** |

In [10]:
print(type(df["Age"]))
print(type(df[["Age"]]))

df.loc[0:4, ["Age", "Sex", "Cholesterol", "HeartDisease"]]

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,Age,Sex,Cholesterol,HeartDisease
0,40,M,289,0
1,49,F,180,1
2,37,M,283,0
3,48,F,214,1
4,54,M,195,0


**Cuidado con `loc` e `iloc`:** `df.loc[0:4]` incluye la fila 4; `df.iloc[0:4]` no. `loc`
trabaja con etiquetas y el rango es cerrado; `iloc` trabaja con posiciones y sigue la convención
de Python.

In [11]:
print("loc[0:4] devuelve", len(df.loc[0:4]), "filas")
print("iloc[0:4] devuelve", len(df.iloc[0:4]), "filas")

loc[0:4] devuelve 5 filas
iloc[0:4] devuelve 4 filas


## 4. Filtrado por condición

Una condición sobre una Series devuelve una máscara booleana, y esa máscara indexa el
DataFrame. Las condiciones se combinan con `&` (y), `|` (o) y `~` (no), **cada una entre
paréntesis**.

In [12]:
mascara = (df["Age"] > 60) & (df["HeartDisease"] == 1)
print(f"{mascara.sum()} pacientes mayores de 60 años con enfermedad")

df.loc[mascara, ["Age", "Sex", "MaxHR", "Oldpeak"]].head()

161 pacientes mayores de 60 años con enfermedad


,Age,Sex,MaxHR,Oldpeak
36,65,M,87,1.5
82,63,M,115,0.0
85,66,M,94,1.0
86,65,M,112,2.0
100,65,M,115,1.0


Corresponde implementar una función de filtrado. Es un envoltorio delgado, pero fuerza a
manejar la máscara con explícitud.

In [13]:
def filtrar(df: pd.DataFrame, columna: str, minimo: float, maximo: float) -> pd.DataFrame:
    """Devuelve las filas cuyo valor en `columna` cae en el intervalo cerrado [minimo, maximo].

    Parámetros
    ----------
    df : DataFrame de entrada.
    columna : nombre de una columna numérica.
    minimo, maximo : extremos del intervalo, incluidos.

    Devuelve
    --------
    DataFrame con las filas seleccionadas y todas las columnas originales.
    Los faltantes de `columna` no se seleccionan.
    """
    return df.loc[df[columna].between(minimo, maximo)].copy() # USO COPY PARA NO MODIFICAR LOS DATOS DEL DATAFRAME ORIGINAL


In [14]:
# VERIFICACIÓN
adultos = filtrar(df, "Age", 40, 50)
assert adultos["Age"].between(40, 50).all(), "quedaron filas fuera del intervalo"
assert len(adultos) == int(df["Age"].between(40, 50).sum()), "faltan o sobran filas"
print(f"OK — {len(adultos)} pacientes entre 40 y 50 años")

OK — 236 pacientes entre 40 y 50 años


## 5. Columnas derivadas

Una columna nueva se crea asignando sobre el DataFrame. Para clasificar por tramos,
`pd.cut` es la herramienta directa.

In [15]:
df["riesgo_hr"] = np.where(df["MaxHR"] < 120, "bajo", "normal")

df["grupo_edad"] = pd.cut(
    df["Age"],
    bins=[0, 40, 55, 70, 120],
    labels=["hasta 40", "41-55", "56-70", "más de 70"],
)

df[["Age", "grupo_edad", "MaxHR", "riesgo_hr"]].head()

,Age,grupo_edad,MaxHR,riesgo_hr
0,40,hasta 40,172,normal
1,49,41-55,156,normal
2,37,hasta 40,98,bajo
3,48,41-55,108,bajo
4,54,41-55,122,normal


**`SettingWithCopyWarning`.** Asignar sobre un recorte (`sub = df[df.Age > 60]` y después
`sub["x"] = ...`) avisa que no se sabe si se modifica el original o una copia. La forma correcta
es pedir la copia explícitamente: `sub = df.loc[df.Age > 60].copy()`.

In [16]:
sub = df.loc[df["Age"] > 60].copy()
sub["edad_relativa"] = sub["Age"] - df["Age"].mean()
sub[["Age", "edad_relativa"]].head(3)

,Age,edad_relativa
36,65,11.489107
82,63,9.489107
85,66,12.489107


## 6. Agrupar y resumir

`groupby` parte el DataFrame según una clave y aplica una función de resumen a cada grupo. Es
la operación que más se usa en una auditoría: casi toda pregunta interesante es «¿cómo cambia
esto **según** aquello?».

In [17]:
df.groupby("ChestPainType")["HeartDisease"].mean().sort_values(ascending=False)

ChestPainType
ASY    0.790323
TA     0.434783
NAP    0.354680
ATA    0.138728
Name: HeartDisease, dtype: float64

In [18]:
df.groupby("Sex").agg(
    n=("Age", "size"),
    edad_media=("Age", "mean"),
    colesterol_mediano=("Cholesterol", "median"),
    tasa_enfermedad=("HeartDisease", "mean"),
).round(2)

,n,edad_media,colesterol_mediano,tasa_enfermedad
Sex,,,,
F,193,52.49,243.0,0.26
M,725,53.78,219.0,0.63


Corresponde implementar el resumen por grupo. La media y la mediana juntas anticipan la
comparación de la clase: cuando difieren mucho, hay asimetría o valores extremos.

In [19]:
def resumen_por_grupo(df: pd.DataFrame, grupo: str, variable: str) -> pd.DataFrame:
    """Resume `variable` dentro de cada nivel de `grupo`.

    Parámetros
    ----------
    df : DataFrame de entrada.
    grupo : nombre de una columna categórica.
    variable : nombre de una columna numérica.

    Devuelve
    --------
    DataFrame indexado por los niveles de `grupo`, con las columnas
    ["n", "media", "mediana", "desvio"], ordenado de mayor a menor mediana.
    """
    return df.groupby(grupo)[variable].agg(n='size',media='mean',mediana='median',desvio='std').sort_values('mediana',ascending=False) # EL AGG SE USA PARA 
    # HACER VARIAS OPERACIONES JUNTAS A LA VEZ , AL PONER n= 'size' LO QUE ESTAMOS HACIENDO ES QUE SE VA A CREAR UNA NUEVA COLUMNA LLAMADA n DONDE SE HARA LA OPERACION 
    # INDICADA, QUE EN ESTE CASO ES CONTAR TODAS LAS FILAS DE CADA GRUPO, Y ASI CON CADA UNO.

In [20]:
# VERIFICACIÓN
r = resumen_por_grupo(df, "ChestPainType", "MaxHR")
assert list(r.columns) == ["n", "media", "mediana", "desvio"], "columnas o su orden"
assert r["n"].sum() == len(df), "los grupos no cubren todas las filas"
assert r["mediana"].is_monotonic_decreasing, "falta ordenar por mediana descendente"
print(r.round(2))


tasadeenf= df.groupby('ChestPainType')['HeartDisease'].mean().sort_values(ascending=False)
print(tasadeenf*100)

                 n   media  mediana  desvio
ChestPainType                              
ATA            173  150.21    152.0   22.28
NAP            203  143.24    147.0   25.61
TA              46  147.89    145.0   23.13
ASY            496  128.48    128.0   23.48
ChestPainType
ASY    79.032258
TA     43.478261
NAP    35.467980
ATA    13.872832
Name: HeartDisease, dtype: float64


### Para analizar

1. ¿Qué tipo de dolor de pecho concentra la mayor tasa de enfermedad?
2. En `MaxHR` por tipo de dolor, ¿la media y la mediana cuentan la misma historia en todos los
   grupos? ¿En cuál se separan más, y qué sugiere esa separación?

**Respuesta:**

1. COMO VEMOS, EL TIPO DE DOLOR DE PECHO CON MAYOR TASA DE ENFERMEDAD ES EL ASY COMO ASI LO DEMUESTRA EL CALCULO DE LA TASA DE ENFERMEDAD.
2. EN CASI TODOS LOS GRUPOS SI, EXCEPTUANDO SI SE QUIERE EN NAP, DONDE LA DIFERENCIA ENTRE UNA Y OTRA ES DE APROX 3.76 LPM LO QUE MUESTRA UNA LEVE TENDENCIA O ASIMETRIA HACIA LA IZQUIERDA , POSIBLEMENETE GENERADA POR ALGUNOS VALORES MAS BAJOS DE MAXHR QUE PUEDAN ESTAR INFLUYENDO EN LA MEDIA.

## 7. Tablas cruzadas y conteos

Para dos categóricas, `crosstab` da la tabla de contingencia; con `normalize` da proporciones.

In [21]:
pd.crosstab(df["ChestPainType"], df["HeartDisease"])

HeartDisease,0,1
ChestPainType,,
ASY,104,392
ATA,149,24
NAP,131,72
TA,26,20


In [22]:
pd.crosstab(df["ChestPainType"], df["HeartDisease"], normalize="index").round(3)

HeartDisease,0,1
ChestPainType,,
ASY,0.210,0.790
ATA,0.861,0.139
NAP,0.645,0.355
TA,0.565,0.435


## 8. Ordenar, contar y valores únicos

In [23]:
print(df["ST_Slope"].value_counts())
print()
print("Niveles distintos de ChestPainType:", df["ChestPainType"].nunique())
print("Cuáles:", sorted(df["ChestPainType"].unique()))
print()
df.sort_values("Cholesterol", ascending=False).head(3)[["Age", "Sex", "Cholesterol"]]

ST_Slope
Flat    460
Up      395
Down     63
Name: count, dtype: int64

Niveles distintos de ChestPainType: 4
Cuáles: ['ASY', 'ATA', 'NAP', 'TA']



,Age,Sex,Cholesterol
149,54,M,603
616,67,F,564
76,32,M,529


## 9. Faltantes y tipos

Este conjunto no trae `NaN`, pero sí trae faltantes **disfrazados de cero**. Es el caso más
frecuente en datos reales, y el motivo por el que `isna().sum()` nunca alcanza como única
revisión.

In [24]:
print("Faltantes declarados por columna:")
print(df.isna().sum().to_string())

sin_codigo = df.replace({"Cholesterol": {0: np.nan}, "RestingBP": {0: np.nan}})
print()
print("Faltantes después de reconocer los códigos:")
print(sin_codigo.isna().sum().to_string())

Faltantes declarados por columna:
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
riesgo_hr         0
grupo_edad        0

Faltantes después de reconocer los códigos:
Age                 0
Sex                 0
ChestPainType       0
RestingBP           1
Cholesterol       172
FastingBS           0
RestingECG          0
MaxHR               0
ExerciseAngina      0
Oldpeak             0
ST_Slope            0
HeartDisease        0
riesgo_hr           0
grupo_edad          0


### DECIDE

`Cholesterol == 0` aparece en 172 registros. Hay tres caminos: dejarlo, convertirlo a `NaN`, o
eliminar esas filas.

1. ¿Cuál corresponde en esta etapa del trabajo y por qué?
2. ¿Qué información se pierde con cada uno de los otros dos?

**Respuesta:**

1.  YO LOS CONVERTIRIA A NAN , PORQUE PRIMERO QUE EL COLESTEROL NO PUEDE SER 0 , POR LO QUE SE ESTA INDICANDO UN VALOR FALTANTE, Y LUEGO TAMBIEN PORQUE SI LOS DEJO EN 0 , ME VAN A ALTERAR RESULTADOS DE VALORES COMO LA MEDIA ENTRE OTRAS COSAS. 
2. SI LOS DEJARAMOS COMO 0 SE PERDERIA EL VALOR REAL DE LAS MEDICIONES COMO LA MEDIA, Y SI LOS ELIMINARAMOS , ESTARIAMOS PERDIEINDO DATOS VALOSIOSO DEL DATAFRAME COMO POR EJEMPLO OTROS DATOS ASOCIADOS A ESTOS PACIENTES QUE PUDIERAN SER VALIOSOS.

## 10. Guardar el resultado

Toda transformación intermedia se guarda con un nombre que dice qué se hizo. `index=False`
evita que el índice se escriba como una columna anónima que reaparece en la próxima carga.

In [25]:
sin_codigo.to_csv("heart_sin_codigos.csv", index=False)
control = pd.read_csv("heart_sin_codigos.csv")
print(control.shape, "— faltantes:", int(control.isna().sum().sum()))

(918, 14) — faltantes: 173


## Cierre

- `read_csv` es donde se decide el tipo de cada columna. Un identificador leído como entero ya
  llega dañado a la auditoría.
- `describe()` es la primera herramienta de detección de errores: un mínimo de 0 en una variable
  fisiológica es un código de error, no una medición.
- `groupby` responde la pregunta que importa en una auditoría: cómo cambia una variable **según**
  otra. Es la base de lo que sigue.

El laboratorio siguiente usa exactamente estas operaciones sobre el conjunto de datos del curso.